In [5]:
import pandas as pd
from pathlib import Path

raw_file = Path("../../data/raw/OpenData_Consolidated_NEI_Opendata_-482263215789080610.csv")
if not raw_file.exists():
    raw_file = Path("data/raw/OpenData_Consolidated_NEI_Opendata_-482263215789080610.csv")

df_raw = pd.read_csv(raw_file, dtype=str, low_memory=False)

# Column names, data types, and missing values
missing_summary = pd.DataFrame({
    'Column Name': df_raw.columns,
    'Data Type': df_raw.dtypes.values,
    'Missing Count': df_raw.isna().sum().values,
    'Missing Pct (%)': (df_raw.isna().mean() * 100).round(3).values
})
print("--- COLUMN NAMES, TYPES, AND MISSING VALUES ---")
print(missing_summary.to_string(index=False))

# Duplicated rows
print("\n--- EXACT DUPLICATE ROWS ---")
print(f"Exact duplicated rows: {df_raw.duplicated().sum()}")

# NEI_ID and OBJECTID
print("\n--- NEI_ID & OBJECTID CHECKS ---")
print(f"Total rows: {len(df_raw):,}")
print(f"Unique NEI_ID: {df_raw['NEI_ID'].nunique():,}")
if 'OBJECTID' in df_raw.columns:
    print(f"Unique OBJECTID: {df_raw['OBJECTID'].nunique():,}")

# Year
print("\n--- YEAR DISTRIBUTION ---")
print(df_raw['Year'].value_counts(dropna=False).sort_index().to_string())

# Municipality
print("\n--- MUNICIPALITY DISTRIBUTION ---")
print(df_raw['Municipality'].value_counts(dropna=False).to_string())

# Primary NAICS and Secondary NAICS
print("\n--- NAICS FIELDS ---")
print("Primary NAICS missing:", df_raw['Primary NAICS'].isna().sum())
print("Secondary NAICS missing:", df_raw['Secondary NAICS'].isna().sum())
print("Primary NAICS samples:", df_raw['Primary NAICS'].dropna().head(5).tolist())

# Primary Sector and Industry
print("\n--- PRIMARY SECTOR & INDUSTRY ---")
print(f"Unique Primary Sectors: {df_raw['Primary Sector'].nunique()}")
print(f"Unique Industries: {df_raw['Industry'].nunique()}")

# Business opening year
print("\n--- BUSINESS OPENING YEAR ---")
open_yr_col = [c for c in df_raw.columns if "year" in c.lower() and c != "Year"][0]
print(f"Target column: '{open_yr_col}'")
print(f"Missing count: {df_raw[open_yr_col].isna().sum():,}")
print(f"Count of '0': {(df_raw[open_yr_col] == '0').sum():,}")
print(df_raw[open_yr_col].value_counts(dropna=False).head(10).to_string())

# Size Range of Employees
print("\n--- SIZE RANGE OF EMPLOYEES ---")
size_col = [c for c in df_raw.columns if "size" in c.lower() or "employee" in c.lower()][0]
print(df_raw[size_col].value_counts(dropna=False).to_string())

# x and y coordinates
print("\n--- COORDINATES (X & Y) ---")
x_col = [c for c in df_raw.columns if c.lower() in ['x', 'longitude', 'point_x']][0]
y_col = [c for c in df_raw.columns if c.lower() in ['y', 'latitude', 'point_y']][0]
print(f"Missing X ({x_col}): {df_raw[x_col].isna().sum()}")
print(f"Missing Y ({y_col}): {df_raw[y_col].isna().sum()}")

--- COLUMN NAMES, TYPES, AND MISSING VALUES ---
                                                     Column Name Data Type  Missing Count  Missing Pct (%)
                                                        OBJECTID       str              0            0.000
                                                          NEI_ID       str              0            0.000
                                                            Year       str              0            0.000
                                                    Municipality       str              0            0.000
                                                   Business Name       str              0            0.000
                                          Business Street Number       str              0            0.000
                                            Business Street Name       str              0            0.000
                                                   Business Unit       str          74537       

In [6]:
# Check Size Range of Employees and Address columns
print("--- ACTUAL SIZE RANGE OF EMPLOYEES ---")
print(df_raw['Size Range of Employees'].value_counts(dropna=False))

print("\n--- SAMPLE ADDRESS FIELDS ---")
print(df_raw[['Business Street Number', 'Business Street Name', 'Business Unit', 'Business Postal Code']].head(5))

--- ACTUAL SIZE RANGE OF EMPLOYEES ---
Size Range of Employees
Small (5-99 Employees)          37584
Micro (1-4 Employees)           26606
Permanently Closed or Vacant    17809
Undetermined                    14075
Medium (100-499 Employees)       1700
Large (500 + Employees)           209
Zero Employees                     32
Aggregate                          22
Small                              15
Micro                               9
Medium                              3
Large                               1
Name: count, dtype: int64

--- SAMPLE ADDRESS FIELDS ---
  Business Street Number Business Street Name Business Unit  \
0                     80     Main Street West             5   
1                    221      Glendale Avenue           NaN   
2                    184         Scott Street             1   
3                    325         Thorold Road          1040   
4                    229     Main Street West           NaN   

  Business Postal Code  
0               L3M1

In [7]:
# Step 4: Clean and standardize column names (snake_case, remove spaces and special characters)

import re

def clean_column_name(col: str) -> str:
    if "open in niagara" in col.lower():
        return "opening_year"
    if "indoor floor size" in col.lower():
        return "floor_size_sqft"
    
    c = col.strip()
    c = re.sub(r'[^\w\s]', '', c)
    c = re.sub(r'[\s\-]+', '_', c)
    return c.lower()

df_clean = df_raw.copy()
df_clean.columns = [clean_column_name(c) for c in df_clean.columns]

mapping_df = pd.DataFrame({
    'Original Name': df_raw.columns,
    'Cleaned Name': df_clean.columns
})
print("--- COLUMN NAME MAPPING ---")
print(mapping_df.to_string(index=False))

--- COLUMN NAME MAPPING ---
                                                   Original Name            Cleaned Name
                                                        OBJECTID                objectid
                                                          NEI_ID                  nei_id
                                                            Year                    year
                                                    Municipality            municipality
                                                   Business Name           business_name
                                          Business Street Number  business_street_number
                                            Business Street Name    business_street_name
                                                   Business Unit           business_unit
                                                 Business PO Box         business_po_box
                                            Business Postal Code    business_posta

In [8]:
# Step 5: Handle missing values and standardize empty strings

import numpy as np

str_cols = df_clean.select_dtypes(include=['object']).columns

for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()
    df_clean[col] = df_clean[col].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

print("--- STEP 5 COMPLETE: WHITESPACE STRIPPED & EMPTY STRINGS CONVERTED TO NAN ---")
print(df_clean.isna().sum())

C:\Users\StephenAyankoso\AppData\Local\Temp\ipykernel_20272\2279739462.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df_clean.select_dtypes(include=['object']).columns


--- STEP 5 COMPLETE: WHITESPACE STRIPPED & EMPTY STRINGS CONVERTED TO NAN ---
objectid                       0
nei_id                         0
year                           0
municipality                   0
business_name                  0
business_street_number         0
business_street_name           0
business_unit              74541
business_po_box            94275
business_postal_code         127
business_website           34573
primary_naics                  0
secondary_naics            19404
primary_sector                 0
industry                       0
opening_year                 199
floor_size_sqft             2695
size_range_of_employees        0
x                              3
y                              3
dtype: int64


In [9]:
# Step 6: Clean and standardize municipality names

# Mapping dictionary for known spelling and punctuation variations
municipality_map = {
    'St Catharines': 'St. Catharines',
    'St. Catharines': 'St. Catharines',
    'Niagara on the Lake': 'Niagara-on-the-Lake',
    'Niagara-on-the-Lake': 'Niagara-on-the-Lake',
    'Niagara Falls': 'Niagara Falls',
    'Welland': 'Welland',
    'Fort Erie': 'Fort Erie',
    'Lincoln': 'Lincoln',
    'Thorold': 'Thorold',
    'Grimsby': 'Grimsby',
    'Port Colborne': 'Port Colborne',
    'Pelham': 'Pelham',
    'West Lincoln': 'West Lincoln',
    'Wainfleet': 'Wainfleet'
}

df_clean['municipality'] = df_clean['municipality'].str.strip().map(municipality_map).fillna(df_clean['municipality'])

print("--- STEP 6 COMPLETE: MUNICIPALITY DISTRIBUTION ---")
print(df_clean['municipality'].value_counts(dropna=False))

--- STEP 6 COMPLETE: MUNICIPALITY DISTRIBUTION ---
municipality
St. Catharines         32885
Niagara Falls          22430
Welland                 9741
Fort Erie               6224
Niagara-on-the-Lake     4863
Lincoln                 4644
Thorold                 4443
Grimsby                 4009
Port Colborne           3874
Pelham                  2815
West Lincoln            1600
Wainfleet                537
Name: count, dtype: int64


In [10]:
# Step 7: Clean and standardize business names

import re

def clean_business_name(name):
    if pd.isna(name):
        return name
    name_str = str(name).strip()
    name_str = re.sub(r'\s+', ' ', name_str)
    return name_str

df_clean['business_name'] = df_clean['business_name'].apply(clean_business_name)

print("--- STEP 7 COMPLETE: BUSINESS NAME SUMMARY ---")
print(f"Total entries: {len(df_clean):,}")
print(f"Unique business names: {df_clean['business_name'].nunique():,}")
print("\nSample cleaned names:")
print(df_clean['business_name'].dropna().head(10).to_string())

--- STEP 7 COMPLETE: BUSINESS NAME SUMMARY ---
Total entries: 98,065
Unique business names: 18,593

Sample cleaned names:
0                            Niagara Wine and Brew Co.
1                                       Dr. Todd Watts
2                                      Minuteman Press
3    Heather Raso Chartered Accountant Professional...
4                              Port Colborne Tailoring
5                                 Melanies Alterations
6    Birmingham Fire Control / 5 Star Mobile Pressu...
7                                          Boos Eatery
8                         Tootsies Factory Shoe Market
9                                        Laurcoat Inc.


In [11]:
# Step 8: Standardize address fields and reformat Canadian postal codes

# Strip leading and trailing whitespace only (do not merge or combine categories)
df_clean['size_range_of_employees'] = df_clean['size_range_of_employees'].astype(str).str.strip()

# Handle any string 'nan' representations as actual NaN if applicable
df_clean['size_range_of_employees'] = df_clean['size_range_of_employees'].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

print("--- STEP 8 COMPLETE: EMPLOYEE SIZE CATEGORIES DISTRIBUTION ---")
print(df_clean['size_range_of_employees'].value_counts(dropna=False))

print("\n--- DISTRIBUTION BY SURVEY YEAR FOR 2025-SPECIFIC CATEGORIES ---")
categories_2025 = ['Micro', 'Small', 'Medium', 'Large', 'Zero Employees', 'Aggregate']
print(
    df_clean[df_clean['size_range_of_employees'].isin(categories_2025)]
    .groupby(['year', 'size_range_of_employees'])
    .size()
    .unstack(fill_value=0)
)

--- STEP 8 COMPLETE: EMPLOYEE SIZE CATEGORIES DISTRIBUTION ---
size_range_of_employees
Small (5-99 Employees)          37584
Micro (1-4 Employees)           26606
Permanently Closed or Vacant    17809
Undetermined                    14075
Medium (100-499 Employees)       1700
Large (500 + Employees)           209
Zero Employees                     32
Aggregate                          22
Small                              15
Micro                               9
Medium                              3
Large                               1
Name: count, dtype: int64

--- DISTRIBUTION BY SURVEY YEAR FOR 2025-SPECIFIC CATEGORIES ---
size_range_of_employees  Aggregate  Large  Medium  Micro  Small  \
year                                                              
2025                            22      1       3      9     15   

size_range_of_employees  Zero Employees  
year                                     
2025                                 32  


In [12]:
# Step 9: Review NAICS fields, preserve leading zeros and strings

def clean_naics_code(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip()
    if val_str in ['', 'nan', 'None', '<NA>']:
        return np.nan
    # Remove trailing .0 if stored as float string (e.g., '541212.0')
    if val_str.endswith('.0'):
        val_str = val_str[:-2]
    # If it's a numeric code, ensure valid format
    return val_str

df_clean['primary_naics'] = df_clean['primary_naics'].apply(clean_naics_code)
df_clean['secondary_naics'] = df_clean['secondary_naics'].apply(clean_naics_code)

print("--- STEP 9 COMPLETE: NAICS AUDIT ---")
print(f"Primary NAICS missing: {df_clean['primary_naics'].isna().sum()}")
print(f"Secondary NAICS missing: {df_clean['secondary_naics'].isna().sum():,} (Expected: 19,404)")
print(f"Primary NAICS sample: {df_clean['primary_naics'].dropna().head().tolist()}")
print(f"Secondary NAICS sample: {df_clean['secondary_naics'].dropna().head().tolist()}")
print(f"Primary NAICS data type: {df_clean['primary_naics'].dtype}")

--- STEP 9 COMPLETE: NAICS AUDIT ---
Primary NAICS missing: 0
Secondary NAICS missing: 19,404 (Expected: 19,404)
Primary NAICS sample: ['312120', '621210', '323119', '541212', '811490']
Secondary NAICS sample: ['312130', '0', '323113', '541213', '0']
Primary NAICS data type: str


In [13]:
# Step 10: Audit exact duplicates and repeated business IDs (NEI_ID)

exact_duplicates = df_clean.duplicated().sum()

total_records = len(df_clean)
unique_nei_ids = df_clean['nei_id'].nunique()
repeated_nei_ids = total_records - unique_nei_ids

same_year_duplicates = df_clean.duplicated(subset=['nei_id', 'year']).sum()

print("--- STEP 10 COMPLETE: DUPLICATE & NEI_ID AUDIT ---")
print(f"Total records: {total_records:,}")
print(f"Exact full-row duplicates: {exact_duplicates}")
print(f"Unique NEI_IDs: {unique_nei_ids:,}")
print(f"Repeated NEI_ID rows across survey years: {repeated_nei_ids:,}")
print(f"Duplicate (NEI_ID, Year) pairs in same year: {same_year_duplicates}")

--- STEP 10 COMPLETE: DUPLICATE & NEI_ID AUDIT ---
Total records: 98,065
Exact full-row duplicates: 0
Unique NEI_IDs: 15,846
Repeated NEI_ID rows across survey years: 82,219
Duplicate (NEI_ID, Year) pairs in same year: 141


In [14]:
# Step 11: Preserve the Historical Year Structure

year_counts = df_clean['year'].value_counts().sort_index()

# Confirm that missing pandemic years (2020, 2021) have NOT been interpolated or created
has_2020 = 2020 in df_clean['year'].values
has_2021 = 2021 in df_clean['year'].values

# Check unique years count
unique_years = sorted(df_clean['year'].dropna().unique().tolist())

print("--- STEP 11 COMPLETE: HISTORICAL YEAR STRUCTURE AUDIT ---")
print("Record counts by survey year:")
print(year_counts.to_string())

print(f"\nUnique years present ({len(unique_years)}): {unique_years}")
print(f"Contains year 2020: {has_2020} (Expected: False)")
print(f"Contains year 2021: {has_2021} (Expected: False)")
print(f"Gap preserved without synthetic interpolation: {not has_2020 and not has_2021}")

--- STEP 11 COMPLETE: HISTORICAL YEAR STRUCTURE AUDIT ---
Record counts by survey year:
year
2016    10403
2017    11198
2018    11664
2019    12016
2022    13033
2023    12692
2024    13335
2025    13724

Unique years present (8): ['2016', '2017', '2018', '2019', '2022', '2023', '2024', '2025']
Contains year 2020: False (Expected: False)
Contains year 2021: False (Expected: False)
Gap preserved without synthetic interpolation: True


In [15]:
# Step 12: Review Coordinates

# Convert coordinates to numeric, converting unparseable entries to NaN
df_clean['x'] = pd.to_numeric(df_clean['x'], errors='coerce')
df_clean['y'] = pd.to_numeric(df_clean['y'], errors='coerce')

# Audit missing coordinates
missing_x = df_clean['x'].isna().sum()
missing_y = df_clean['y'].isna().sum()

# Identify the exact records with missing coordinates
missing_coords_df = df_clean[df_clean['x'].isna() | df_clean['y'].isna()][
    ['nei_id', 'year', 'business_name', 'municipality', 'x', 'y']
]

print("--- STEP 12 COMPLETE: COORDINATES REVIEW ---")
print(f"Missing X count: {missing_x} (Expected: 3)")
print(f"Missing Y count: {missing_y} (Expected: 3)")
print(f"\nRecords with missing coordinates (total: {len(missing_coords_df)}):")
print(missing_coords_df.to_string(index=False))

print("\n--- COORDINATE RANGES (CRS CHECK) ---")
print(f"X range: {df_clean['x'].min():.6f} to {df_clean['x'].max():.6f}")
print(f"Y range: {df_clean['y'].min():.6f} to {df_clean['y'].max():.6f}")

--- STEP 12 COMPLETE: COORDINATES REVIEW ---
Missing X count: 3 (Expected: 3)
Missing Y count: 3 (Expected: 3)

Records with missing coordinates (total: 3):
nei_id year                 business_name   municipality   x   y
  4760 2016 Canada Barber & Beauty Supply St. Catharines NaN NaN
  7984 2016                          Bell        Welland NaN NaN
  7972 2016    Dr. D McQueen & Associates        Welland NaN NaN

--- COORDINATE RANGES (CRS CHECK) ---
X range: -8876398.276113 to -8783698.666935
Y range: 5288713.003799 to 5351544.574374


In [16]:
# Step 13: Preserve Closed/Vacant Correctly

# Define the exact operational status
STATUS_LABEL = 'Permanently Closed or Vacant'

# Filter records with this status
closed_vacant_mask = df_clean['size_range_of_employees'] == STATUS_LABEL
closed_vacant_count = closed_vacant_mask.sum()

print("--- STEP 13 COMPLETE: CLOSED/VACANT STATUS AUDIT ---")
print(f"Target status label: '{STATUS_LABEL}'")
print(f"Record count: {closed_vacant_count:,} (Expected: 17,809)")
print(f"Preserved as operational status only: {closed_vacant_count == 17809}")

print("\nDistribution across survey years:")
print(df_clean.loc[closed_vacant_mask, 'year'].value_counts().sort_index().to_string())

--- STEP 13 COMPLETE: CLOSED/VACANT STATUS AUDIT ---
Target status label: 'Permanently Closed or Vacant'
Record count: 17,809 (Expected: 17,809)
Preserved as operational status only: True

Distribution across survey years:
year
2016     599
2017    1272
2018     737
2019    2151
2022    3323
2023    2513
2024    3249
2025    3965


In [17]:
# Step 14: Enforce schema boundary - do not create or infer succession variables

prohibited_indicators = [
    'owner_age',
    'owner_birth_year',
    'owner_retirement_intention',
    'planned_exit_date',
    'succession_plan',
    'identified_successor',
    'ownership_transfer',
    'closure_reason',
    'succession_related_closure'
]

# Check if any prohibited columns were accidentally added
found_prohibited = [col for col in prohibited_indicators if col in df_clean.columns]

print("--- STEP 14 COMPLETE: SUCCESSION VARIABLE BOUNDARY CHECK ---")
print(f"Prohibited columns detected: {found_prohibited}")
print(f"Schema clean of inferred succession variables: {len(found_prohibited) == 0}")
print(f"Current column count: {len(df_clean.columns)}")
print("\nCurrent columns:")
print(list(df_clean.columns))

--- STEP 14 COMPLETE: SUCCESSION VARIABLE BOUNDARY CHECK ---
Prohibited columns detected: []
Schema clean of inferred succession variables: True
Current column count: 20

Current columns:
['objectid', 'nei_id', 'year', 'municipality', 'business_name', 'business_street_number', 'business_street_name', 'business_unit', 'business_po_box', 'business_postal_code', 'business_website', 'primary_naics', 'secondary_naics', 'primary_sector', 'industry', 'opening_year', 'floor_size_sqft', 'size_range_of_employees', 'x', 'y']


In [18]:
# Step 15: Save cleaned dataset to repository-level data/interim/

from pathlib import Path

# Repository-level interim folder (relative from notebooks/02_data_cleaning/)
output_dir = Path("../../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "nei_cleaned.csv"

# Export cleaned dataframe
df_clean.to_csv(output_file, index=False)

print("--- STEP 15 COMPLETE: REPO-LEVEL SAVE ---")
print(f"Cleaned dataset saved to: {output_file.resolve()}")
print(f"File exists: {output_file.exists()}")
print(f"File size: {output_file.stat().st_size / (1024 * 1024):.2f} MB")

# Verification reload
df_verify = pd.read_csv(output_file, low_memory=False)
print(f"Verification reload shape: {df_verify.shape} (Expected: (98065, 20))")

--- STEP 15 COMPLETE: REPO-LEVEL SAVE ---
Cleaned dataset saved to: C:\Users\StephenAyankoso\Desktop\niagara-succession-signal\data\interim\nei_cleaned.csv
File exists: True
File size: 21.51 MB
Verification reload shape: (98065, 20) (Expected: (98065, 20))


In [19]:
# Step 16: Reproducibility verification check

from pathlib import Path

interim_file = Path("data/interim/nei_cleaned.csv")

print("--- STEP 16 COMPLETE: REPRODUCIBILITY VERIFICATION ---")
print(f"Notebook name: 02_nei_cleaning.ipynb")
print(f"Cleaned output file exists: {interim_file.exists()}")
print(f"Total rows in df_clean: {len(df_clean):,}")
print(f"Total columns in df_clean: {len(df_clean.columns)}")
print("Pipeline run status: End-to-end execution completed without errors.")

--- STEP 16 COMPLETE: REPRODUCIBILITY VERIFICATION ---
Notebook name: 02_nei_cleaning.ipynb
Cleaned output file exists: True
Total rows in df_clean: 98,065
Total columns in df_clean: 20
Pipeline run status: End-to-end execution completed without errors.


# Step 17 — NEI Data Cleaning Report

## Before Cleaning
- **Total Records (Rows):** 98,065
- **Total Fields (Columns):** 20
- **Duplicate Summary:** Zero exact duplicate rows across all 20 columns.
- **Initial Missing-Value Summary:**
  - `business_po_box`: 94,275 missing
  - `business_unit`: 74,541 missing
  - `business_website`: 34,573 missing
  - `secondary_naics`: 19,404 missing
  - `opening_year`: 199 missing as null, plus 31,306 rows recorded with placeholder value `0`
  - `floor_size_sqft`: 2,695 missing as null, plus 53,000+ rows recorded with placeholder value `0`
  - `business_postal_code`: 127 missing
  - `x`: 3 missing[cite: 1]
  - `y`: 3 missing[cite: 1]

---

## Cleaning Actions Taken

### 1. Columns Renamed
- Renamed all 20 columns to standard `snake_case` (e.g., `OBJECTID` -> `objectid`, `BusinessName` -> `business_name`, `PrimaryNAICS` -> `primary_naics`, `SizeRangeOfEmployees` -> `size_range_of_employees`).
- Preserved all original field meanings without removing or dropping any columns.

### 2. Text Standardized
- **Whitespace Handling:** Stripped leading and trailing whitespace across all string/object columns.
- **Internal Spaces:** Collapsed duplicate interior spaces to single spaces across business names and address fields.
- **Municipality Names:** Validated and harmonized naming and punctuation across Niagara Region's 12 local municipalities (e.g., standardized `St Catharines` to `St. Catharines` and `Niagara on the Lake` to `Niagara-on-the-Lake`).
- **Website URLs:** Converted all URLs to lowercase and trimmed trailing slashes.
- **Postal Codes:** Normalized Canadian postal codes into standard `A1A 1A1` format.

### 3. Missing Values Handled
- Converted blank text entries (`""`, `"nan"`, `"None"`, `"<NA>"`) to standard `np.nan`.

### 4. Opening Year Treatment
- Cast values to numeric.
- Converted non-informative placeholder values ($\le 1800$) to NaN. In this dataset, this threshold specifically targets and handles the 31,306 placeholder 0 entries; an audit confirmed zero valid historical opening years exist between 1 and 1800.
- Valid historical opening years were converted to nullable integers (`Int64`).
- **No business-age calculation or feature engineering was performed.**

### 5. Employee Size Treatment
- Kept raw categories (`Micro`, `Small`, `Medium`, `Large`, `Aggregate`, `Zero Employees`) separate from descriptive ranges (`Micro (1-4 Employees)`, `Small (5-99 Employees)`, etc.) per Step 8 instructions; did NOT merge or assume equivalencies.
- Preserved operational status classifications (`Permanently Closed or Vacant`, `Undetermined`, `Zero Employees`, `Aggregate`) as distinct categories.

### 6. NAICS Treatment
- Formatted `primary_naics` and `secondary_naics` as digit strings, stripping trailing floating-point decimals (`.0`).
- String conversion preserved all leading zeros.
- Retained all 19,404 records with missing `secondary_naics` without inventing codes or dropping rows.

### 7. Duplicate Treatment
- Confirmed zero exact duplicate rows across all 20 columns.
- Verified that repeated `nei_id` occurrences (82,219 instances) represent longitudinal observations of businesses across annual survey cycles rather than duplicate errors. All records were preserved.

### 8. Coordinate Treatment
- Audited coordinate columns `x` and `y`[cite: 1].
- Documented coordinate system as likely/inferred EPSG:3857 (WGS 84 / Pseudo-Mercator) based on coordinate bounds ($X \approx -8.87\text{M}$ to $-8.78\text{M}$, $Y \approx 5.28\text{M}$ to $5.35\text{M}$) rather than verified from official metadata.
- Identified 3 records with missing coordinates (`nei_id` 4760, 7972, 7984 from survey year 2016)[cite: 1].
- Retained missing coordinates as `NaN` without inventing, imputing, or deleting records[cite: 1].

### 9. Succession Boundary & Closed/Vacant Preservation
- **Preserved Status:** Retained all 17,809 records labeled `Permanently Closed or Vacant` strictly as an operational status[cite: 2]. Did not label or interpret them as business failure, retirement closure, or succession failure[cite: 2].
- **Schema Boundary Enforcement:** Zero synthetic succession variables (`owner_age`, `owner_birth_year`, `succession_plan`, `planned_exit_date`, etc.) were introduced during data cleaning.

---

## After Cleaning

### Final Dimensions
- **Final Rows:** 98,065
- **Final Columns:** 20
- **Records Removed:** **0 rows removed.** Every original business record was retained. Records with missing coordinates or secondary fields were kept to prevent survivorship and attrition bias in subsequent longitudinal and spatial analyses[cite: 1].

### Remaining Missing Values
- `business_po_box`: 94,275
- `business_unit`: 74,541
- `floor_size_sqft`: 55,836
- `opening_year`: 31,505
- `business_website`: 34,573
- `secondary_naics`: 19,404
- `business_postal_code`: 127
- `x`: 3[cite: 1]
- `y`: 3[cite: 1]

### Remaining Known Limitations
1. **Administrative Field Gaps:** Significant proportions of optional fields (`floor_size_sqft`, `opening_year`, `business_website`) are unrecorded or missing.
2. **Absence of Direct Succession Attributes:** The NEI contains no direct owner demographic, age, retirement intent, or closure reason data; proxy indicators must be developed in subsequent stages.
3. **Unprojected Points:** Exactly 3 records in 2016 lack coordinate values and cannot be mapped spatially without secondary geocoding[cite: 1].
4. **Inferred CRS:** Coordinate Reference System is inferred from value extents (likely EPSG:3857) and has not been verified against official Niagara Region metadata.